In [2]:
from dotenv import load_dotenv
load_dotenv()
from getpass import getpass

def deterministic_guardrails(text: str) -> bool:
    """Return true if content if blocked"""
    banned_kw=["hack","exploit","malware","bomb"]
    return any(kw in text.lower() for kw in banned_kw)

text_inputs=[
    'what should i do to avoid hack a database?',
    'what is the capital of France?',
    'explain how to spread a malware?'
]

for inp in text_inputs:
    blocked=deterministic_guardrails(inp)
    status="Blocked" if blocked else "Allow"
    print(f'{status}:{inp}')
    

Blocked:what should i do to avoid hack a database?
Allow:what is the capital of France?
Blocked:explain how to spread a malware?


In [13]:
from langchain_openai import ChatOpenAI

def model_based_gurdrails(text: str) -> str:
    """Use LLM to evaluate content safety, Return SAFETY and UNSAFE"""
    model=ChatOpenAI(model='gpt-4o-mini')
    prompt=f"""Is the following use input safe to process? Reply with only 'SAFE or 'UNSAFE"""

    result=model.invoke([{'role':'user','content':prompt}])
    return result.content.strip()

text=[
    "what should i do to avoid hack a database?",
    "what is the capital of France?",
    "explain how to spread a Malware virus?"
]

for inp in text:
    verdict=model_based_gurdrails(inp)
    status='UNSAFE' if 'UNSAFE' in verdict else "SAFE"
    print(f'{status}:{inp}')

SAFE:what should i do to avoid hack a database?
SAFE:what is the capital of France?
SAFE:explain how to spread a Malware virus?


In [16]:
## BUILD INBUILD GUARDRAILS
#PII personal identifiable information detection middleware

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def customer_lookup(query:str)->str:
    """Look up customer information"""
    return f'Customer record found for query: {query}'

agents=create_agent(model='gpt-3.5-turbo', tools=[customer_lookup],
                    middleware=[PIIMiddleware('email',strategy="redact",apply_to_input=True),
                                PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
                                PIIMiddleware("api_key",detector=r"sk-[a-zA-Z0-9](32)", strategy="block", apply_to_input=True)])

result=agents.invoke({"messages":[{"role":"user","content":"My email is john.doe@example.com and my card is 2345-3233-2424-4422, can you help me"}]})


In [17]:
print(result['messages'][-1].content)

I have found customer records associated with the email [REDACTED_EMAIL] and the card number 2345-3233-2424-4422. How can I assist you further?
